In [ ]:
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from langchain_teddynote.messages import stream_response
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

from dotenv import load_dotenv
import os

# API KEY 정보로드
load_dotenv()

# 프로젝트 이름을 입력합니다.
logging.langsmith(os.environ["STUDENT_NAME"] + "-" + "Simple_Chat")

# ChatOpenAI 객체를 생성합니다.
gpt = ChatOpenAI(
    temperature=1,  # 창의성 (0.0 ~ 2.0)
    model_name="gpt-4o-mini",  # 올바른 모델명으로 수정
)



In [ ]:

# 1. 세션별 대화 이력을 저장할 딕셔너리 생성
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """세션 ID에 해당하는 대화 이력을 반환하거나 새로 생성합니다."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 2. GPT 모델에 대화 이력 관리 기능(RunnableWithMessageHistory)을 래핑합니다.
with_message_history = RunnableWithMessageHistory(
    gpt,
    get_session_history,
)



In [ ]:

# 3. 세션 설정 (동일한 session_id를 사용하면 이전 대화를 기억합니다)
config = {"configurable": {"session_id": "user_session_1"}}

# --- 첫 번째 질문 (주식 매수 문의) ---
question1 = "삼성전자 주식을 매수 해도 되나요?"
print(f"[질문 1] {question1}\n[답변 1] ", end="")

# 이력이 포함된 객체로 스트리밍 실행
answer1 = with_message_history.stream(question1, config=config)
stream_response(answer1)
print("\n" + "="*50 + "\n")



In [ ]:

# --- 두 번째 질문 (이전 대화 기억 확인) ---
question2 = "내가 방금 전에 어떤 기업에 대해 물어봤었지?"
print(f"[질문 2] {question2}\n[답변 2] ", end="")

# 동일한 config(session_id)를 전달하여 이전 문맥을 기억하게 합니다.
answer2 = with_message_history.stream(question2, config=config)
stream_response(answer2)
print()